In [ ]:
# =======================================================================
# UAH-DRIVESET PROCESSOR (ROBUST DIAGNOSTIC VERSION)
# Author: Piranavan Sathiyavannan (Adapted by Gemini)
# Description:
#    ✅ Handles different CSV separators (comma, semicolon, space).
#    ✅ Prints detailed debug info to find why events are missed.
#    ✅ Ensures labels are applied correctly.
# =======================================================================

import pandas as pd
import numpy as np
import os
import glob
import sys

# --- CONFIGURATION ---
RAW_DATA_DIR = "raw_data"
OUTPUT_FILE = "real_world_validation.csv"
TARGET_FREQ = "20ms" # 50Hz

# MAPPING: UAH Label -> CARLA Label
LABEL_MAP = {
    'evento_nao_agressivo': 'Normal_Driving',
    'freada_agressiva': 'Harsh_Brake',
    'aceleracao_agressiva': 'Sudden_Acceleration',
    'curva_esquerda_agressiva': 'Sharp_Turn',
    'curva_direita_agressiva': 'Sharp_Turn',
    'troca_faixa_esquerda_agressiva': 'Sudden_Lane_Change',
    'troca_faixa_direita_agressiva': 'Sudden_Lane_Change'
}

def load_ground_truth(gt_path):
    """Attempts to load groundTruth with various separators."""
    separators = [';', ',', '\t', '\s+']
    
    for sep in separators:
        try:
            if sep == '\s+':
                df = pd.read_csv(gt_path, delim_whitespace=True, engine='python')
            else:
                df = pd.read_csv(gt_path, sep=sep, engine='python')
            
            # Clean column names (strip whitespace)
            df.columns = [c.strip().lower() for c in df.columns]
            
            # Check if valid columns exist
            if 'evento' in df.columns and 'inicio' in df.columns:
                return df
                
        except Exception:
            continue
            
    return pd.DataFrame() # Empty if all fail

def process_folder(folder_path):
    folder_name = os.path.basename(folder_path)
    print(f"\n📂 Processing Trip: {folder_name}...")

    try:
        # 1. Locate Files
        acc_path = os.path.join(folder_path, "acelerometro_terra.csv")
        gyro_path = os.path.join(folder_path, "giroscopio_terra.csv")
        gt_path = os.path.join(folder_path, "groundTruth.csv")

        if not (os.path.exists(acc_path) and os.path.exists(gyro_path)):
            print(f"   ⚠️ Missing sensor files. Skipping.")
            return None

        # 2. Load Sensors
        df_acc = pd.read_csv(acc_path)
        df_gyro = pd.read_csv(gyro_path)
        
        # 3. Load Ground Truth (With Robust Loader)
        df_gt = load_ground_truth(gt_path)
        
        if df_gt.empty:
            print(f"   ⚠️ Could not read groundTruth.csv. Skipping labels.")
        else:
            print(f"   ✅ Found {len(df_gt)} events in groundTruth.")

        # 4. Time Sync & Resample
        # Base time = first accelerometer timestamp
        start_t = df_acc['uptimeNanos'].iloc[0]
        
        df_acc['seconds'] = (df_acc['uptimeNanos'] - start_t) / 1e9
        df_gyro['seconds'] = (df_gyro['uptimeNanos'] - start_t) / 1e9
        
        df_acc.index = pd.to_timedelta(df_acc['seconds'], unit='s')
        df_gyro.index = pd.to_timedelta(df_gyro['seconds'], unit='s')
        
        # Resample to 50Hz
        acc_res = df_acc[['x', 'y', 'z']].resample(TARGET_FREQ).mean().interpolate()
        gyro_res = df_gyro[['x', 'y', 'z']].resample(TARGET_FREQ).mean().interpolate()
        
        # Merge
        merged = pd.merge_asof(acc_res, gyro_res, left_index=True, right_index=True, suffixes=('_acc', '_gyro'))
        merged.columns = ['accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z']
        
        # 5. Apply Labels
        merged['event_type'] = 'Normal_Driving'
        merged['event_intensity'] = 'N/A'
        merged['time_sec'] = merged.index.total_seconds()
        
        events_found = 0
        
        if not df_gt.empty:
            for _, row in df_gt.iterrows():
                raw_event = row['evento']
                start = float(row['inicio'])
                end = float(row['fim'])
                
                # Check Mapping
                if raw_event in LABEL_MAP:
                    target_label = LABEL_MAP[raw_event]
                    
                    # Apply to range
                    mask = (merged['time_sec'] >= start) & (merged['time_sec'] <= end)
                    if mask.sum() > 0:
                        merged.loc[mask, 'event_type'] = target_label
                        merged.loc[mask, 'event_intensity'] = 'Real_World'
                        events_found += 1
                        # print(f"      -> Mapped '{raw_event}' to '{target_label}' ({mask.sum()} rows)")
        
        print(f"   ✅ Labeled {events_found} distinct events in this trip.")
        return merged.dropna()

    except Exception as e:
        print(f"   ❌ Error processing {folder_name}: {e}")
        return None

def main():
    print(f"--- STARTING ROBUST PROCESSOR ---")
    
    if not os.path.exists(RAW_DATA_DIR):
        print(f"❌ Folder '{RAW_DATA_DIR}' not found.")
        return

    all_data = []
    folders = [f for f in os.listdir(RAW_DATA_DIR) if os.path.isdir(os.path.join(RAW_DATA_DIR, f))]
    
    for folder in folders:
        df = process_folder(os.path.join(RAW_DATA_DIR, folder))
        if df is not None:
            all_data.append(df)
    
    if not all_data:
        print("❌ No data processed.")
        return

    final_df = pd.concat(all_data)
    if 'time_sec' in final_df.columns: del final_df['time_sec']
    
    # Pseudo-timestamp for compatibility
    final_df['timestamp'] = pd.date_range(start='2025-01-01', periods=len(final_df), freq=TARGET_FREQ)
    final_df = final_df.set_index('timestamp')
    
    final_df.to_csv(OUTPUT_FILE)

    print("\n" + "="*50)
    print(f"✅ DONE! Saved {len(final_df)} rows to '{OUTPUT_FILE}'")
    print("\n--- FINAL DATASET STATS ---")
    print(final_df['event_type'].value_counts())
    print("="*50)
    
    if len(final_df['event_type'].unique()) > 1:
        print("🎉 Success! You have both Normal and Event data.")
        print("👉 Now run 'sim_to_real_validation.py'")
    else:
        print("⚠️ WARNING: Only 'Normal_Driving' found. Check your groundTruth.csv files.")

if __name__ == "__main__":
    main()